# 8 - Relevance Feedback dan Query Expansion

**Tujuan praktikum**
1. Mahasiswa memahami masalah *vocabulary mismatch* dan cara mengatasinya dengan relevance feedback serta query expansion.
2. Mahasiswa mampu mengimplementasikan algoritma **Rocchio** untuk *explicit relevance feedback*.
3. Mahasiswa mampu mengimplementasikan **pseudo relevance feedback** (PRF) dan query expansion berbasis tesaurus di atas BM25.
4. Mahasiswa mampu mengukur perubahan MAP sebelum dan sesudah expansion serta mengenali *query drift*.

**Rumus Rocchio**

$$\vec q\,' = \alpha\,\vec q + \beta\,\frac{1}{|D_r|}\sum_{\vec d \in D_r}\vec d \;-\; \gamma\,\frac{1}{|D_{nr}|}\sum_{\vec d \in D_{nr}}\vec d$$

dengan $D_r$ dokumen relevan, $D_{nr}$ dokumen tidak relevan, dan nilai umum $\alpha = 1$, $\beta = 0{,}75$, $\gamma = 0{,}15$.

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
from stki_utils import (corpus_texts, preprocess, TfidfModel, BM25Model,
                        load_queries, load_qrels, evaluate, average_precision)

docs = {d: preprocess(t) for d, t in corpus_texts().items()}
queries, qrels = load_queries(), load_qrels()
bm25 = BM25Model(docs)       # untuk pemeringkatan
tfidf = TfidfModel(docs)     # vektor dokumen ternormalisasi untuk Rocchio

run_awal = {q: [d for d, _ in bm25.search(preprocess(t), k=10)] for q, t in queries.items()}
evaluate(run_awal, qrels, k=5).loc[["Rata-rata"]]

,P@5,R@5,F1@5,AP,nDCG@5
query_id,,,,,
Rata-rata,0.5,0.8125,0.6131,0.7756,0.8668


## Langkah 1 - Query berbobot pada BM25
Setelah expansion, setiap term query memiliki bobot. Skor BM25 dihitung sebagai $\sum_t w_t \cdot \text{BM25}(t, d)$. Fungsi ini tersedia sebagai `BM25Model.search_weighted`.

In [2]:
def cari_berbobot(q_bobot, k=10):
    return [d for d, _ in bm25.search_weighted(q_bobot, k=k)]

# query tanpa expansion: setiap term berbobot 1, hasilnya sama dengan search biasa
q2 = preprocess(queries["Q2"])
print(cari_berbobot({t: 1.0 for t in q2})[:5])
print(run_awal["Q2"][:5])

['D10', 'D12', 'D06', 'D27', 'D08']
['D10', 'D12', 'D06', 'D27', 'D08']


## Langkah 2 - Implementasi Rocchio
Vektor dokumen diambil dari `TfidfModel` (TF-IDF ternormalisasi). Term dengan bobot negatif dibuang, lalu hanya `m` term baru dengan bobot terbesar yang ditambahkan agar query tidak terlalu panjang.

In [4]:
def centroid(doc_ids):
    c = defaultdict(float)
    for d in doc_ids:
        for t, w in tfidf.doc_vectors[d].items():
            c[t] += w / len(doc_ids)
    return c

def rocchio(q_tokens, relevan, tidak_relevan=(), alpha=1.0, beta=0.75, gamma=0.15, m=5):
    q = defaultdict(float)
    for t in q_tokens:
        q[t] += alpha
    if relevan:
        for t, w in centroid(relevan).items():
            q[t] += beta * w
    if tidak_relevan:
        for t, w in centroid(tidak_relevan).items():
            q[t] -= gamma * w
    asli = {t: w for t, w in q.items() if t in q_tokens and w > 0}
    baru = sorted(((t, w) for t, w in q.items() if t not in q_tokens and w > 0), key=lambda x: -x[1])[:m]
    return {**asli, **dict(baru)}

q_baru = rocchio(q2, relevan=["D10", "D09"])
pd.Series(q_baru, name="bobot").round(3).to_frame()

,bobot
teknologi,1.060
tani,1.095
modern,1.000
hidroponik,0.135
tanah,0.122
cocok,0.119
air,0.107
manfaat,0.095


## Langkah 3 - Explicit relevance feedback (simulasi)
Pada explicit RF, pengguna menandai hasil awal. Di sini penilaian pengguna disimulasikan dengan qrels: dari 5 hasil teratas BM25, dokumen yang ada di qrels dianggap relevan dan sisanya tidak relevan.

In [5]:
def explicit_rf(q, k=5, **kw):
    top = run_awal[q][:k]
    rel = [d for d in top if d in qrels[q]]
    nonrel = [d for d in top if d not in qrels[q]]
    return rocchio(preprocess(queries[q]), rel, nonrel, **kw)

run_explicit = {q: cari_berbobot(explicit_rf(q)) for q in queries}
print("Query Q2 setelah explicit RF:", {t: round(w, 3) for t, w in explicit_rf("Q2").items()})
evaluate(run_explicit, qrels, k=5).loc[["Rata-rata"]]

Query Q2 setelah explicit RF: {'teknologi': 1.043, 'tani': 1.096, 'modern': 1.0, 'pupuk': 0.17, 'organik': 0.125, 'tanah': 0.117, 'kompos': 0.11, 'buat': 0.099}


,P@5,R@5,F1@5,AP,nDCG@5
query_id,,,,,
Rata-rata,0.525,0.8542,0.6443,0.8638,0.905


## Langkah 4 - Pseudo (blind) relevance feedback
Tanpa bertanya ke pengguna, sistem menganggap `k` dokumen teratas relevan, lalu menambahkan `m` term terbaik dari dokumen tersebut. Ini adalah bentuk query expansion **lokal**.

In [6]:
def prf(q, k=3, m=5):
    return rocchio(preprocess(queries[q]), run_awal[q][:k], m=m)

run_prf = {q: cari_berbobot(prf(q)) for q in queries}
pd.DataFrame([(q, queries[q], ", ".join(t for t in prf(q) if t not in preprocess(queries[q])))
              for q in queries], columns=["query_id", "query", "term tambahan"])

,query_id,query,term tambahan
0,Q1,pencegahan penyakit demam berdarah,"olahraga, aktivitas, fisik, kadar, diabetes"
1,Q2,teknologi pertanian modern,"dapat, gabah, jaga, aplikasi, telemedisin"
2,Q3,gizi anak balita,"sekolah, stunting, tumbuh, tuju, status"
3,Q4,mesin pencari dan indeks dokumen,"mungkin, pustaka, temu, katalog, kata"
4,Q5,pemasaran digital UMKM,"usaha, kecil, kredit, bank, rakyat"
5,Q6,keamanan data pribadi di internet,"layan, komputasi, awan, telemedisin, pasien"
6,Q7,harga hasil panen petani,"kopi, gabah, padi, tingkat, raya"
7,Q8,pembelajaran daring mahasiswa,"akses, beasiswa, digital, pustaka, temu"


In [7]:
evaluate(run_prf, qrels, k=5).loc[["Rata-rata"]]

,P@5,R@5,F1@5,AP,nDCG@5
query_id,,,,,
Rata-rata,0.525,0.8542,0.6443,0.8208,0.8898


## Langkah 5 - Query expansion global dengan tesaurus
Sumber global tidak bergantung pada hasil pencarian. Contoh berikut memakai tesaurus kecil buatan sendiri (sinonim/kata terkait). Pada praktiknya dapat memakai Tesaurus Tematis Bahasa Indonesia, Kateglo, atau *word embedding* (Word2Vec/FastText). Term tambahan diberi bobot lebih kecil (0,5) daripada term asli.

In [8]:
TESAURUS = {
    "modern": ["teknologi", "sensor", "otomatis"],
    "cegah": ["vaksin", "imunisasi"],
    "daring": ["internet", "digital"],
    "pasar": ["jual", "marketplace"],
    "aman": ["sandi", "phishing"],
    "panen": ["gabah", "beras"],
}

def tesaurus_qe(q, bobot=0.5):
    q_tokens = preprocess(queries[q])
    hasil = {t: 1.0 for t in q_tokens}
    for t in q_tokens:
        for s in TESAURUS.get(t, []):
            for s_stem in preprocess(s):
                hasil.setdefault(s_stem, bobot)
    return hasil

run_tesaurus = {q: cari_berbobot(tesaurus_qe(q)) for q in queries}
print("Q2:", tesaurus_qe("Q2"))
evaluate(run_tesaurus, qrels, k=5).loc[["Rata-rata"]]

Q2: {'teknologi': 1.0, 'tani': 1.0, 'modern': 1.0, 'sensor': 0.5, 'otomatis': 0.5}


,P@5,R@5,F1@5,AP,nDCG@5
query_id,,,,,
Rata-rata,0.5,0.8125,0.6131,0.7937,0.8718


## Langkah 6 - Perbandingan sebelum dan sesudah expansion

In [9]:
ringkasan = pd.DataFrame({
    "BM25 (tanpa expansion)": evaluate(run_awal, qrels, k=5).loc["Rata-rata"],
    "BM25 + tesaurus (global)": evaluate(run_tesaurus, qrels, k=5).loc["Rata-rata"],
    "BM25 + PRF (k=3, m=5)": evaluate(run_prf, qrels, k=5).loc["Rata-rata"],
    "BM25 + Rocchio explicit": evaluate(run_explicit, qrels, k=5).loc["Rata-rata"],
}).T.rename(columns={"AP": "MAP"})
ringkasan

,P@5,R@5,F1@5,MAP,nDCG@5
BM25 (tanpa expansion),0.500,0.8125,0.6131,0.7756,0.8668
BM25 + tesaurus (global),0.500,0.8125,0.6131,0.7937,0.8718
"BM25 + PRF (k=3, m=5)",0.525,0.8542,0.6443,0.8208,0.8898
BM25 + Rocchio explicit,0.525,0.8542,0.6443,0.8638,0.9050


In [9]:
ax = ringkasan[["MAP", "nDCG@5"]].plot.barh(figsize=(7, 3.5))
ax.set_xlim(0, 1); ax.set_xlabel("skor"); ax.set_title("Pengaruh relevance feedback dan query expansion")
ax.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()

## Langkah 7 - Analisis per query dan query drift
Rata-rata bisa menutupi query yang justru memburuk. Hitung selisih AP per query antara PRF dan BM25 awal.

In [10]:
per_query = pd.DataFrame({
    "query": queries,
    "AP awal": {q: average_precision(run_awal[q], set(qrels[q])) for q in queries},
    "AP PRF": {q: average_precision(run_prf[q], set(qrels[q])) for q in queries},
})
per_query["selisih"] = per_query["AP PRF"] - per_query["AP awal"]
per_query.round(4)

,query,AP awal,AP PRF,selisih
Q1,pencegahan penyakit demam berdarah,0.7500,0.7500,0.0000
Q2,teknologi pertanian modern,0.6179,0.6179,0.0000
Q3,gizi anak balita,0.6667,0.6667,0.0000
Q4,mesin pencari dan indeks dokumen,0.5556,0.9167,0.3611
Q5,pemasaran digital UMKM,1.0000,1.0000,0.0000
Q6,keamanan data pribadi di internet,0.8095,0.8095,0.0000
Q7,harga hasil panen petani,1.0000,1.0000,0.0000
Q8,pembelajaran daring mahasiswa,0.8056,0.8056,0.0000


Perhatikan bahwa kenaikan MAP PRF pada korpus ini seluruhnya berasal dari satu query (Q4). Lihat juga term tambahan Q2 pada Langkah 4: dua dari tiga dokumen teratas Q2 (D12 dan D06) tidak relevan, sehingga PRF menambahkan term seperti `gabah` dan `telemedisin` yang tidak berhubungan dengan teknologi pertanian.

Jika dokumen teratas hasil awal tidak relevan, PRF menambahkan term dari dokumen yang salah sehingga query "hanyut" menjauhi maksud pengguna (**query drift**). Sel berikut menguji sensitivitas PRF terhadap jumlah dokumen feedback `k` dan jumlah term tambahan `m`.

In [11]:
grid = {}
for k in [1, 2, 3, 5, 10]:
    for m in [3, 5, 10, 20]:
        run = {q: cari_berbobot(prf(q, k=k, m=m)) for q in queries}
        grid[(k, m)] = evaluate(run, qrels, k=5).loc["Rata-rata", "AP"]
tabel = pd.Series(grid).unstack()
tabel.index.name, tabel.columns.name = "k dokumen", "m term"
print("MAP BM25 tanpa expansion:", evaluate(run_awal, qrels, k=5).loc["Rata-rata", "AP"])
tabel

MAP BM25 tanpa expansion: 0.7756


m term,3,5,10,20
k dokumen,,,,
1,0.7895,0.7895,0.7895,0.8645
2,0.7756,0.8047,0.7890,0.8047
3,0.7756,0.8208,0.8208,0.8029
5,0.7822,0.8208,0.8208,0.8187
10,0.7760,0.7993,0.8062,0.8125


## Tugas 8
1. Evaluasi explicit RF di atas memberi keuntungan tidak adil karena dokumen yang sudah dinilai pengguna ikut dihitung. Terapkan evaluasi **residual collection**: buang 5 dokumen yang sudah dinilai dari hasil dan dari qrels, lalu bandingkan kembali BM25 awal dan Rocchio explicit.
2. Ubah nilai $\gamma$ (0; 0,15; 0,5) dan $\beta$ (0,5; 0,75; 1,0). Bagaimana pengaruhnya terhadap MAP?
3. Ganti cara pemilihan term PRF dari bobot Rocchio menjadi frekuensi term di dokumen feedback. Mana yang lebih baik, dan mengapa?
4. Perluas tesaurus minimal 20 entri dan ukur ulang MAP. Temukan satu contoh sinonim yang justru menurunkan kinerja (ambiguitas) dan jelaskan.
5. (Opsional) Latih Word2Vec atau FastText pada korpus yang lebih besar (misalnya korpus proyek akhir) dan gunakan kata terdekat sebagai sumber expansion.

In [12]:
# Tulis jawaban Anda di sini